##Import libraries and define the source

In [0]:
from getpass import getpass
import requests
import pandas as pd

storage_account = "energystrgacc"
bronze_container = "bronze"

blob_path = "weather/adf_test/location=chennai/forecast.json"

print("Notebook setup completed.")
print("Source container:", bronze_container)
print("Source file:", blob_path)

##Read the raw JSON from Bronze

In [0]:
bronze_sas = getpass("Enter Bronze container SAS token: ").strip().lstrip("?")

bronze_url = (
    f"https://{storage_account}.blob.core.windows.net/"
    f"{bronze_container}/{blob_path}?{bronze_sas}"
)

response = requests.get(bronze_url, timeout=30)

print("HTTP status:", response.status_code)

response.raise_for_status()

weather_json = response.json()

print("Top-level JSON keys:", list(weather_json.keys()))

##Extract the hourly weather records

In [0]:
hourly_data = weather_json["hourly"]

weather_pandas_df = pd.DataFrame(hourly_data)

weather_pandas_df["location"] = "chennai"

print("Rows loaded:", len(weather_pandas_df))
print("Columns:", weather_pandas_df.columns.tolist())

display(weather_pandas_df.head())

##Convert the records into a Spark DataFrame

In [0]:
weather_spark_df = spark.createDataFrame(weather_pandas_df)

print("Spark row count:", weather_spark_df.count())

weather_spark_df.printSchema()

display(weather_spark_df.limit(5))

##Define the Silver transformation

In [0]:
from pyspark.sql import functions as F

silver_df = weather_spark_df.select(
    F.col("location").cast("string").alias("location"),
    F.to_timestamp(F.col("time")).alias("weather_timestamp"),
    F.col("temperature_2m").cast("double").alias("temperature_2m"),
    F.col("relative_humidity_2m").cast("double").alias("relative_humidity_2m"),
    F.col("precipitation").cast("double").alias("precipitation"),
    F.col("wind_speed_10m").cast("double").alias("wind_speed_10m")
)

display(silver_df.limit(5))
silver_df.printSchema()

##Handle invalid records and duplicates

In [0]:
silver_df = silver_df.filter(
    F.col("location").isNotNull()
    & F.col("weather_timestamp").isNotNull()
)

silver_df = silver_df.dropDuplicates(
    ["location", "weather_timestamp"]
)

print("Rows after cleaning:", silver_df.count())

##Add ingestion metadata

In [0]:
silver_df = (
    silver_df
    .withColumn("ingested_at_utc", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
)

display(silver_df.limit(5))
silver_df.printSchema()

##Validate the Silver data

In [0]:
print("Total Silver rows:", silver_df.count())

print(
    "Distinct location/timestamp keys:",
    silver_df.select(
        "location", "weather_timestamp"
    ).distinct().count()
)

print("Rows with null temperature:",
      silver_df.filter(F.col("temperature_2m").isNull()).count())

print("Rows with null precipitation:",
      silver_df.filter(F.col("precipitation").isNull()).count())

display(
    silver_df.orderBy("location", "weather_timestamp").limit(10)
)

##Save the Silver Delta table in Databricks

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_hourly_silver")

print("Silver Delta table saved.")

In [0]:
saved_silver_df = spark.table("weather_hourly_silver")

print("Saved row count:", saved_silver_df.count())
saved_silver_df.printSchema()

display(saved_silver_df.limit(10))

##Parquet writer

In [0]:
#%pip install pyarrow

In [0]:
#%restart_python

##Inspect the pandas DataFrame

In [0]:
# Display each column's data type
print(silver_pandas_df.dtypes)

# Identify columns containing unexpected Python objects
for column in silver_pandas_df.columns:
    sample_types = (
        silver_pandas_df[column]
        .dropna()
        .map(lambda value: type(value).__name__)
        .unique()
    )
    print(column, "->", sample_types)

##Convert Silver data to Parquet

In [0]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

parquet_path = "/tmp/weather_hourly_silver.parquet"

# Convert the pandas DataFrame into a PyArrow table
arrow_table = pa.Table.from_pandas(
    silver_pandas_df,
    preserve_index=False
)

# Write the table directly to a Parquet file
pq.write_table(arrow_table, parquet_path)

# Verify the file
print("File exists:", os.path.exists(parquet_path))
print("File size (bytes):", os.path.getsize(parquet_path))
print("Rows written:", pq.read_metadata(parquet_path).num_rows)

##Upload the Parquet file to Azure Storage

In [0]:
from getpass import getpass
import requests
import os

# Azure Storage details
storage_account = "energystrgacc"
silver_container = "silver"
blob_name = "weather_hourly/weather_hourly_silver.parquet"

# Enter the Silver container SAS token privately
silver_sas = getpass("Enter Silver container SAS token: ").strip().lstrip("?")

# Build the upload URL
upload_url = (
    f"https://{storage_account}.blob.core.windows.net/"
    f"{silver_container}/{blob_name}?{silver_sas}"
)

# Read the Parquet file
with open("/tmp/weather_hourly_silver.parquet", "rb") as file:
    parquet_bytes = file.read()

# Upload the file as an Azure block blob
response = requests.put(
    upload_url,
    headers={
        "x-ms-blob-type": "BlockBlob",
        "Content-Type": "application/octet-stream"
    },
    data=parquet_bytes,
    timeout=60
)

print("Upload HTTP status:", response.status_code)
response.raise_for_status()
print("Parquet file uploaded successfully.")
print("Bytes uploaded:", len(parquet_bytes))

##Verify the uploaded file

In [0]:
from getpass import getpass
import requests

silver_sas = getpass("Enter Silver container SAS token: ").strip().lstrip("?")

blob_url = (
    "https://energystrgacc.blob.core.windows.net/"
    "silver/weather_hourly/weather_hourly_silver.parquet"
)

# Append the SAS token unchanged
response = requests.get(
    f"{blob_url}?{silver_sas}",
    headers={"Range": "bytes=0-3"},
    timeout=30
)

print("Azure read status:", response.status_code)
print("Bytes returned:", len(response.content))

if response.status_code in (200, 206):
    print("Azure read verified successfully.")

else:
    # Show the Azure error message, but never print the SAS token.
    print("Azure error:", response.text[:500])

##Merge Upsert

In [0]:
import os
import re
import getpass
import requests
import xml.etree.ElementTree as ET
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==============================================================================
# 1. CONFIGURATION (NO HARDCODING)
# ==============================================================================
# Can be filled via input prompts or environment variables
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT") or input("Storage Account: ").strip()
CONTAINER       = os.getenv("BRONZE_CONTAINER") or input("Container Name: ").strip()
PREFIX          = os.getenv("BLOB_PREFIX") or input("Prefix/Folder (e.g. weather/adf_test): ").strip().strip("/")
TARGET_TABLE    = os.getenv("TARGET_TABLE") or input("Target Silver Table: ").strip()
SAS_TOKEN       = getpass.getpass("Bronze SAS Token: ").strip().lstrip("?")

BASE_CONTAINER_URL = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net/{CONTAINER}"

REQUIRED_METRICS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m"
]

# ==============================================================================
# 2. DISCOVER & PARSE FILES (SINGLE PASS)
# ==============================================================================
weather_records = []
skipped_files = []
marker = None

print("Starting file discovery and extraction...")

while True:
    list_params = {
        "restype": "container",
        "comp": "list",
        "prefix": PREFIX
    }
    if marker:
        list_params["marker"] = marker

    list_resp = requests.get(
        f"{BASE_CONTAINER_URL}?{SAS_TOKEN}",
        params=list_params,
        timeout=60
    )
    list_resp.raise_for_status()

    root = ET.fromstring(list_resp.text)

    for blob in root.findall(".//{*}Blob"):
        blob_path = blob.findtext("{*}Name")
        if not (blob_path and blob_path.lower().endswith("_forecast.json")):
            continue

        # Extract location dynamically from folder structure (e.g., location=chennai/)
        loc_match = re.search(r"(?:^|/)location=([^/]+)(?:/|$)", blob_path, flags=re.IGNORECASE)
        if not loc_match:
            skipped_files.append((blob_path, "Missing 'location=' in path"))
            continue

        location = loc_match.group(1).lower()

        # Fetch JSON content
        file_url = f"{BASE_CONTAINER_URL}/{blob_path}?{SAS_TOKEN}"
        file_resp = requests.get(file_url, timeout=60)
        
        if file_resp.status_code != 200:
            skipped_files.append((blob_path, f"HTTP {file_resp.status_code}"))
            continue

        payload = file_resp.json()
        hourly = payload.get("hourly", {})
        times = hourly.get("time", [])

        # Validate hourly arrays exist and match timestamp length
        if not times or any(len(hourly.get(k, [])) != len(times) for k in REQUIRED_METRICS):
            skipped_files.append((blob_path, "Mismatched or missing hourly fields"))
            continue

        # Flatten records
        for i, timestamp in enumerate(times):
            weather_records.append({
                "location": location,
                "weather_timestamp": timestamp,
                "temperature_2m": float(hourly["temperature_2m"][i]) if hourly["temperature_2m"][i] is not None else None,
                "relative_humidity_2m": int(hourly["relative_humidity_2m"][i]) if hourly["relative_humidity_2m"][i] is not None else None,
                "precipitation": float(hourly["precipitation"][i]) if hourly["precipitation"][i] is not None else None,
                "wind_speed_10m": float(hourly["wind_speed_10m"][i]) if hourly["wind_speed_10m"][i] is not None else None,
            })

    marker = root.findtext(".//{*}NextMarker")
    if not marker:
        break

print(f"Extraction complete: {len(weather_records)} rows extracted, {len(skipped_files)} files skipped.")

# ==============================================================================
# 3. CONVERT TO SPARK DATAFRAME & MERGE TO SILVER DELTA
# ==============================================================================
if weather_records:
    source_df = (
        spark.createDataFrame(weather_records)
        .withColumn("weather_timestamp", F.to_timestamp("weather_timestamp", "yyyy-MM-dd'T'HH:mm"))
        .withColumn("ingested_at_utc", F.current_timestamp())
        .withColumn("ingestion_date", F.current_date())
        .dropDuplicates(["location", "weather_timestamp"])
    )

    # Dynamic change condition: NOT (t.col <=> s.col)
    update_condition = " OR ".join([f"NOT (t.{col} <=> s.{col})" for col in REQUIRED_METRICS])

    (
        DeltaTable.forName(spark, TARGET_TABLE)
        .alias("t")
        .merge(
            source_df.alias("s"),
            "t.location = s.location AND t.weather_timestamp = s.weather_timestamp"
        )
        .whenMatchedUpdate(
            condition=update_condition,
            set={col: f"s.{col}" for col in REQUIRED_METRICS + ["ingested_at_utc", "ingestion_date"]}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Successfully merged into {TARGET_TABLE}.")
else:
    print("No valid records found to merge.")

In [0]:
%sql
select location,weather_timestamp,count(*) from weather_hourly_silver
group by all
having count(*)>1;